# Arbeitspaket (AP) 2: Datenbeschaffung, Datenaufbereitung & explorative Datenanalyse

**Prüfung – 40 Punkte**

### Persönliche Angaben (bitte ergänzen) und Prüfungsregeln

<table>
  <tr><td>Vorname:</td><td></td></tr>
  <tr><td>Nachname:</td><td></td></tr>
  <tr><td>Immatrikulationsnummer:</td><td></td></tr>
  <tr><td>Modul:</td><td>Data Science</td></tr>
  <tr><td>Datum / Zeit:</td><td>09.11.2026 / Prüfungsbeginn 08:00 Uhr</td></tr>
  <tr><td>Erlaubte Hilfsmittel:</td><td>Es gelten die Prüfungsregeln (PDF-Dokument auf EXAM-Moodle).</td></tr>
  <tr><td>Nicht erlaubt:</td><td>Es gelten die Prüfungsregeln (PDF-Dokument auf EXAM-Moodle).</td></tr>
  <tr><td>Technische Probleme:</td><td>Wenden Sie sich bei technischen Problemen umgehend an die Prüfungsaufsicht.</td></tr>
</table>

## Bewertungskriterien

| Teil | Aufgaben | Punkte |
|---|---|---:|
| A: Datenverständnis und Auswertung mit Python | A1–A3 | 12 |
| B: Fehleranalyse und Korrektur | B1–B2 | 16 |
| C: Python-Code schreiben oder ergänzen | C1–C3 | 12 |
| **Total** | **8 Aufgaben** | **40** |

### Bewertungsschema

Jede Aufgabe wird für sich und **als Ganzes** bewertet. Ihre Punktzahl steht in der Überschrift.

<table>
  <thead>
    <tr><th>Kategorie</th><th>Beschreibung</th><th>Bewertung</th><th>4 Pkt</th><th>8 Pkt</th></tr>
  </thead>
  <tbody>
    <tr><td>Korrekt</td><td>Vollständig und fachlich einwandfrei: Antwort bzw. Begründung korrekt und nachvollziehbar; bei Code-Aufgaben liefert der Python-Code das richtige Ergebnis</td><td>100%</td><td>4</td><td>8</td></tr>
    <tr><td>Minimale Mängel</td><td>Kleine Fehler (z.B. falsche Sortierung, fehlende Rundung, fehlende Achsenbeschriftung) oder Begründung ungenau bzw. unvollständig</td><td>75%</td><td>3</td><td>6</td></tr>
    <tr><td>Mittlere Mängel</td><td>Ein wesentlicher Teil der Aufgabe ist korrekt gelöst. Eine weitere wesentliche Anforderung fehlt oder ist fachlich falsch.</td><td>50%</td><td>2</td><td>4</td></tr>
    <tr><td>Gravierende Mängel</td><td>Problem bzw. Ansatz erkennbar, aber Lösung oder Schlussfolgerung im Kern falsch (im Code z.B. ungeeigneter Lösungsweg wie falscher Join-Typ, falsche Tabelle oder falsche Datei)</td><td>25%</td><td>1</td><td>2</td></tr>
    <tr><td>Nicht lauffähig / nicht sinnvoll</td><td>Antwort fehlt, kein Bezug zur Aufgabe, oder Code nicht lauffähig (Syntax- oder Laufzeitfehler) und kein korrekter schriftlicher Teil.</td><td>0%</td><td>0</td><td>0</td></tr>
  </tbody>
</table>

Alternative, fachlich korrekte Lösungswege werden ebenfalls als korrekt gewertet.

## Hinweise zu den Daten

Die fiktive Tierhandlung «PfotenWelt» führt 156 Tierfuttervarianten. Eine Produkt-ID bezeichnet genau eine Packungsvariante. Gleiche Namen können bei verschiedenen Packungsgrössen vorkommen.

Alle Dateien liegen lokal vor:

| Datei | Inhalt |
|---|---|
| `html/shop_page_1.html` bis `shop_page_3.html` | Drei gespeicherte Produktlisten mit zusammen 163 Karten |
| `html/product_detail.html` | Detailseite mit Fütterungsempfehlung und Preisverlauf |
| `pet_food_clean.csv` | Aufbereitete Daten der 156 Produkte |
| `sales.json` | Verkaufte Packungen im Oktober 2026 je Produkt |
| `pet_shop_data.xlsx` | Spaltenbeschreibungen, Daten und Quellenübersicht |

Der Setup liest die HTML-Karten (je Produkt ein Kasten auf der Shop-Seite mit Marke, Name, Tierart, Packung, Preis und Lagerbestand; im HTML ein `<div class="product-card">`) zur Ansicht als Text in `df_raw` ein und lädt die aufbereiteten Daten als `df_clean`. Alle Aufgaben verwenden `df_clean` bzw. die in der Aufgabe genannten Dateien; `df_raw` muss nicht bereinigt werden. Ein Preis gilt je Packung; `pack_g` ist das Gewicht in Gramm. Ein unbekannter Lagerbestand ist nicht gleich null. Die Werte «Preis folgt» bedeuten einen unbekannten Preis.

Öffnen Sie dieses Notebook im Ordner AP02. Es wird ausschliesslich aus lokalen Dateien gelesen, keine Website oder API aufgerufen.

## Setup (vorgegeben, nicht bewertet)

Diese Zellen einmal von oben nach unten ausführen. Alle Daten liegen lokal vor.

In [ ]:
# Benötigte Bibliotheken importieren.
import os
import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup

# Aktuelles Arbeitsverzeichnis anzeigen.
print('Working directory:', os.getcwd())

In [ ]:
# Vorgegeben: drei lokal gespeicherte Shop-Seiten einlesen.
rows = []
for page in [1, 2, 3]:
    with open(f'html/shop_page_{page}.html', encoding='utf-8') as file:
        soup = BeautifulSoup(file.read(), 'html.parser')
    for card in soup.select('div.product-card'):
        rows.append([
            card['data-product-id'],
            card.select('span.brand')[0].get_text().strip(),
            card.select('span.product-name')[0].get_text().strip(),
            card.select('li.animal')[0].get_text().strip(),
            card.select('li.pack')[0].get_text().strip(),
            card.select('span.price')[0].get_text().strip(),
            card.select('span.stock')[0].get_text().strip(),
        ])

# Aus den eingelesenen Produktkarten einen DataFrame erstellen.
df_raw = pd.DataFrame(
    rows,
    columns=[
        'product_id',
        'brand',
        'product_name',
        'animal',
        'pack_raw',
        'price_raw',
        'stock_raw',
    ],
)

# Umfang und erste Zeilen der Rohdaten anzeigen.
print('Rohdaten:', df_raw.shape)
df_raw.head()

In [ ]:
# Aufbereitete Produktdaten aus der lokalen CSV-Datei laden.
df_clean = pd.read_csv('pet_food_clean.csv')

# Umfang und erste Zeilen der bereinigten Daten anzeigen.
print('Bereinigte Produktdaten:', df_clean.shape)
df_clean.head()

---

<div style="background-color:#2E4057; color:white; padding:16px 24px; border-radius:6px; font-size:18px; margin-top:8px;">
&#9654;&nbsp; <strong>Ab hier beginnt die bewertete Prüfung — 40 Punkte</strong><br>
<span style="font-size:15px; font-weight:normal;">Tragen Sie Ihre Antworten direkt in die vorgesehenen Zellen ein und führen Sie alle Code-Zellen aus.</span>
</div>

Die Aufgaben sind unabhängig voneinander lösbar. Verändern Sie die im Setup geladenen DataFrames nicht; verwenden Sie eigene Variablen je Aufgabe. Teil B enthält bewusst fachlich fehlerhaften, aber lauffähigen Code.

**Schriftliche Antworten und Begründungen** tragen Sie jeweils in die Markdown-Zelle «Ihre Antwort / Begründung» unter der Aufgabe ein, Code in die zugehörige Codezelle.

---
## Teil A – Datenverständnis und Auswertung mit Python (12 Punkte)

### A1 (4 Punkte)

Die Tierhandlung erfasst Preise markenweise. Bei der Marke **FutterFreude** war die Erfassung beim Export noch nicht abgeschlossen; bei elf Produkten zeigt der Shop deshalb «Preis folgt». Der Erfassungsstand hängt nicht von der Preishöhe ab.

a) Um welchen Mechanismus fehlender Werte handelt es sich (MCAR, MAR oder MNAR)? Begründen Sie.

b) Ein Praktikant schlägt vor, «Preis folgt» durch CHF 0 zu ersetzen, damit `price_chf` keine fehlenden Werte mehr enthält. Beurteilen Sie den Vorschlag.

**Ihre Antwort / Begründung:**

> [Hier eintragen]

### A2 (4 Punkte)

Berechnen Sie **mit Python** die Interquartile Range (IQR) der bekannten Packungspreise für zwei Packungsgrössenklassen: `klein` (bis einschliesslich 250 g) und `gross` (über 750 g). Ergänzen Sie dazu die beiden `TODO`-Stellen.

Der ausgegebene DataFrame muss `pack_size` und `iqr_chf` (CHF, zwei Dezimalstellen) enthalten, eine Zeile je Packungsgrössenklasse.

Was misst der IQR? Bei welcher Packungsgrössenklasse sind die Packungspreise in CHF stärker gestreut? Begründen Sie mit den berechneten Werten.

**Ihre Antwort / Begründung:**

> [Hier eintragen]

In [ ]:
# Vorgegeben: bekannte Packungspreise der beiden Packungsgrössenklassen auswählen.
small_prices = df_clean.loc[df_clean['pack_g'] <= 250, 'price_chf'].dropna()
large_prices = df_clean.loc[df_clean['pack_g'] > 750, 'price_chf'].dropna()

# IQR für jede Packungsgrössenklasse berechnen.
iqr_small = ... # TODO: IQR für kleine Packungen berechnen
iqr_large = ... # TODO: IQR für grosse Packungen berechnen

# Vorgegeben: DataFrame erstellen und ausgeben.
output_A2 = pd.DataFrame({
    'pack_size': ['klein', 'gross'],
    'iqr_chf': [iqr_small, iqr_large],
})
print(output_A2.round(2).to_string(index=False))

### A3 (4 Punkte)

Erstellen Sie aus `df_clean` mit **`pd.pivot_table()`** das durchschnittliche Packungsgewicht (`pack_g`) und den durchschnittlichen Packungspreis (`price_chf`) je Tierart. Erstellen Sie dazu die Pivot Table `pivot_A3` an der `TODO`-Stelle vollständig.

Der DataFrame `output_A3` muss `animal`, `pack_g` und `price_chf` enthalten: eine Zeile je Tierart, Durchschnittswerte auf zwei Dezimalstellen gerundet.

Eine Kollegin vergleicht die durchschnittlichen Packungspreise und folgert: *«Hundefutter ist bei uns am günstigsten.»*

Nennen und begründen Sie **zwei** fachliche Einwände gegen diese Schlussfolgerung.

**Ihre Antwort / Begründung:**

> [Hier eintragen]

In [ ]:
# Vorgegeben: die für die Pivot Table benötigten Spalten übernehmen.
df_A3 = df_clean[['animal', 'pack_g', 'price_chf']].copy()

# TODO: Pivot Table mit den Durchschnittswerten je Tierart erstellen.
pivot_A3 = ...

# Vorgegeben: Tierart als Spalte übernehmen und Durchschnittswerte ausgeben.
output_A3 = pivot_A3.reset_index()[['animal', 'pack_g', 'price_chf']]
print(output_A3.round(2).to_string(index=False))

---
## Teil B – Fehleranalyse (16 Punkte)

### B1 (8 Punkte)

**Ziel:** Den gesamten Warenwert aller lagernden Packungen je Tierart aus `df_clean` ausgeben. Jede lagernde Packung wird dabei mit ihrem Verkaufspreis je Packung (`price_chf`) bewertet. Verwenden Sie nur Produkte mit bekanntem Preis und bekanntem Lagerbestand.

Der DataFrame muss `animal` und `stock_value_chf` (CHF, zwei Dezimalstellen) enthalten, eine Zeile je Tierart.

Die Filialleiterin ist irritiert: *«Allein im Katzenregal stehen Packungen für viele tausend Franken.»*

**Aufgabe:** Schreiben Sie den korrigierten Code in die Codezelle (die Kommentare des ursprünglichen Codes sind als Gerüst vorgegeben). Begründen Sie jede Änderung.

**Zu korrigierender Python-Code:**
```python
# Produkte mit bekanntem Preis und bekanntem Bestand übernehmen.
df_B1 = df_clean.dropna(subset=['price_chf', 'stock_units']).copy()

# Wert je Produkt berechnen.
df_B1['stock_value_chf'] = df_B1['price_chf'] + df_B1['stock_units']

# Lagerwerte je Tierart ausgeben.
stock_value_by_animal = df_B1[['animal', 'stock_value_chf']].groupby(['animal']).mean()
output_B1 = stock_value_by_animal.reset_index()
print(output_B1.round(2).to_string(index=False))
```

**Ausgabe dieses Python-Codes vor der Korrektur:**

Diese Ausgabe zeigt den fehlerhaften Ausgangszustand, nicht das gesuchte korrekte Ergebnis.

```text
animal  stock_value_chf
  Hund            35.00
 Katze            42.66
 Nager            38.14
```

**Ihre Antwort / Begründung:**

> [Hier eintragen]

In [ ]:
# Produkte mit bekanntem Preis und bekanntem Bestand übernehmen.

# Wert je Produkt berechnen.

# Lagerwerte je Tierart ausgeben.

### B2 (8 Punkte)

**Ziel:** Den Median des Kilopreises je Tierart aus `df_clean` ausgeben. Verwenden Sie alle Produkte mit bekanntem Packungspreis.

Der DataFrame muss `animal` und `median_price_per_kg` (Median des Kilopreises in CHF/kg, zwei Dezimalstellen) enthalten, eine Zeile je Tierart.

Der Einkäufer meint: *«Ein Kilo Futter für ein bis zwei Rappen? Das kann nicht stimmen.»*

**Aufgabe:** Schreiben Sie den korrigierten Code in die Codezelle (die Kommentare des ursprünglichen Codes sind als Gerüst vorgegeben). Begründen Sie jede Änderung.

**Zu korrigierender Python-Code:**
```python
# Daten für die Preisauswertung vorbereiten.
df_B2 = df_clean.dropna().copy()

# Preis bezogen auf das Packungsgewicht berechnen.
df_B2['price_per_kg'] = df_B2['price_chf'] / df_B2['pack_g']

# Median je Tierart ausgeben.
median_by_animal = df_B2[['animal', 'price_per_kg']].groupby(['animal']).median()
output_B2 = median_by_animal.reset_index().rename(columns={'price_per_kg': 'median_price_per_kg'})
print(output_B2.round(2).to_string(index=False))
```

**Ausgabe dieses Python-Codes vor der Korrektur:**

Diese Ausgabe zeigt den fehlerhaften Ausgangszustand, nicht das gesuchte korrekte Ergebnis.

```text
animal  median_price_per_kg
  Hund                 0.01
 Katze                 0.02
 Nager                 0.02
```

**Ihre Antwort / Begründung:**

> [Hier eintragen]

In [ ]:
# Daten für die Preisauswertung vorbereiten.

# Preis bezogen auf das Packungsgewicht berechnen.

# Median je Tierart ausgeben.

---
## Teil C – Python-Code schreiben oder ergänzen (12 Punkte)

### C1 (4 Punkte)

Das Gerüst liest `html/product_detail.html` (Detailseite eines Katzenfutters) ein und überführt eine Tabelle in den DataFrame `df_C1`.

Ergänzen Sie die beiden mit `TODO` markierten Stellen:

1. Wählen Sie die Tabelle **«Fütterungsempfehlung»** aus.
2. Wandeln Sie die Spalte `daily_feed_g` in eine numerische Spalte um.

Der DataFrame `df_C1` muss die numerischen Spalten `weight_kg` (Körpergewicht der Katze) und `daily_feed_g` (Tagesmenge in Gramm) enthalten, eine Zeile je Körpergewicht.

Begründen Sie, wie Sie die richtige Tabelle auswählen und wie Sie die Tagesmenge umwandeln.

*Tipp: Öffnen Sie die HTML-Datei im Editor und sehen Sie sich `df_C1` vor der Umwandlung an.*

**Ihre Antwort / Begründung:**

> [Hier eintragen]

In [ ]:
# Lokale HTML-Datei einlesen.
with open('html/product_detail.html', 'r', encoding='utf-8') as file:
    html_content = file.read()

# HTML-Inhalt mit BeautifulSoup einlesen.
soup = BeautifulSoup(html_content, 'html.parser')

# TODO: Tabelle «Fütterungsempfehlung» auswählen.
table = ...

# Tabellenzeilen auslesen.
rows = []
for row in table.find_all('tr'):
    rows.append([cell.get_text().strip() for cell in row.find_all('td')])

# DataFrame erstellen.
df_C1 = pd.DataFrame(rows, columns=['weight_kg', 'daily_feed_g'])

# Vorgegeben: Kopfzeile entfernen (sie hat keine <td>-Zellen und ergibt eine leere Zeile).
df_C1 = df_C1.dropna().reset_index(drop=True)

# Körpergewicht in Zahlen umwandeln.
df_C1['weight_kg'] = pd.to_numeric(df_C1['weight_kg'])

# TODO: Einheit entfernen und Tagesmenge in Zahlen umwandeln.
df_C1['daily_feed_g'] = ...

# DataFrame ausgeben.
print(df_C1.to_string(index=False))

### C2 (4 Punkte)

Erstellen Sie einen **Grouped Boxplot der bekannten Packungspreise nach Tierart**: je ein Boxplot für Hund, Katze und Nager in derselben Grafik. Beschriften Sie die drei Gruppen und beide Achsen.

a) Ein Kollege folgert aus dem Boxplot: *«Nagerfutter ist typischerweise am teuersten, denn dort liegt der höchste Preis.»* Beurteilen Sie diese Aussage anhand der Grafik.

b) Ein Mitarbeiter möchte alle Packungspreise, die im Boxplot als einzelne Punkte erscheinen, vor weiteren Auswertungen löschen. Beurteilen Sie dieses Vorgehen.

*Tipp: Bei `ax.boxplot()` setzt das Argument `tick_labels` die Gruppenbeschriftungen.*

**Ihre Antwort / Begründung:**

> [Hier eintragen]

In [ ]:
# Produkte mit bekanntem Packungspreis auswählen.
df_C2 = ...

# Packungspreise je Tierart auswählen.
prices_hund = ...
prices_katze = ...
prices_nager = ...

# Abbildung und Achse anlegen.
fig_C2, ax_C2 = plt.subplots(figsize=(5, 3))

# Gruppierten Boxplot zeichnen und Achsen beschriften.
...

# Titel setzen und Grafik anzeigen.
ax_C2.set_title('Packungspreise nach Tierart')
plt.tight_layout()
plt.show()

### C3 (4 Punkte)

Die Datei `sales.json` enthält die im Oktober 2026 verkauften Packungen je Produkt (`units_sold`). Lesen Sie die Datei ein, verbinden Sie sie über `product_id` mit `df_clean` und berechnen Sie die verkauften Packungen je Tierart. Stellen Sie das Ergebnis als **Horizontal Bar Chart** (waagrechte Balken) mit beschrifteten Achsen dar.

a) Für welche Tierart wurden die meisten Packungen verkauft?

b) Die Geschäftsleitung folgert aus der Grafik, dass für diese Tierart auch das meiste Futter (in kg) verkauft wurde. Ist dieser Schluss aus der Grafik zulässig? Begründen Sie.

**Ihre Antwort / Begründung:**

> [Hier eintragen]

In [ ]:
# Verkaufszahlen aus der lokalen JSON-Datei einlesen.

# Verkaufszahlen über product_id mit df_clean verbinden.

# Verkaufte Packungen je Tierart summieren.

# Horizontal Bar Chart erstellen und Achsen beschriften.

---
### Notebook-Footer (vorgegeben)

In [ ]:
# Bibliotheken für die technischen Angaben laden.
import platform
from datetime import datetime

# Python-Version, Betriebssystem und Zeitpunkt ausgeben.
print('Python:', platform.python_version())
print('System:', platform.system())
print('Zeitpunkt:', datetime.now().isoformat(timespec='seconds'))